## Exploratory Data Analysis and Cleanup for Children's Books

In [199]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

In [200]:
DetectorFactory.seed = 42

In [201]:
# paths
load_dotenv()
children_path = os.getenv("CHILDREN_BOOKS")

In [202]:
print(children_path)

../data/books/goodreads_books_children.json


In [203]:
children_df = pd.read_json(children_path, lines=True)

In [204]:
children_df.shape

(124082, 29)

In [205]:
children_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,1599150603,7,[],US,,"[{'count': '56', 'name': 'to-read'}, {'count':...",,false,4.13,B00DU10PUG,[],"Relates in vigorous prose the tale of Aeneas, ...",Paperback,https://www.goodreads.com/book/show/287141.The...,"[{'author_id': '3041852', 'role': ''}]",Yesterday's Classics,162,13,9781599150604,9,,2006,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls
1,1934876569,6,[151854],US,,"[{'count': '515', 'name': 'to-read'}, {'count'...",,false,4.22,,"[948696, 439885, 274955, 12978730, 372986, 216...","To Kara's astonishment, she discovers that a p...",Paperback,https://www.goodreads.com/book/show/6066812-al...,"[{'author_id': '19158', 'role': ''}]",Seven Seas,216,3,9781934876565,3,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...
2,0590417010,193,[],US,eng,"[{'count': '450', 'name': 'to-read'}, {'count'...",,false,4.43,B017RORXNI,"[834493, 452189, 140185, 1897316, 2189812, 424...",In Newbery Medalist Cynthia Rylant's classic b...,Hardcover,https://www.goodreads.com/book/show/89378.Dog_...,"[{'author_id': '5411', 'role': ''}]",Blue Sky Press,40,1,9780590417013,9,,1995,https://www.goodreads.com/book/show/89378.Dog_...,https://images.gr-assets.com/books/1360057676m...,89378,1331,86259,Dog Heaven,Dog Heaven
3,0915190575,4,[],US,,"[{'count': '8', 'name': 'to-read'}, {'count': ...",,false,4.29,,[],,,https://www.goodreads.com/book/show/3209312-mo...,"[{'author_id': '589328', 'role': ''}, {'author...",,,,9780915190577,,,,https://www.goodreads.com/book/show/3209312-mo...,https://s.gr-assets.com/assets/nophoto/book/11...,3209312,11,3242879,"Moths and Mothers, Feathers and Fathers: A Sto...","Moths and Mothers, Feathers and Fathers: A Sto..."
4,1416904999,4,[],US,,"[{'count': '8', 'name': 'to-read'}, {'count': ...",,false,3.57,,[],WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,Board Book,https://www.goodreads.com/book/show/1698376.Wh...,"[{'author_id': '169159', 'role': ''}]",Little Simon,24,1,9781416904991,6,,2005,https://www.goodreads.com/book/show/1698376.Wh...,https://s.gr-assets.com/assets/nophoto/book/11...,1698376,23,1695373,What Do You Do?,What Do You Do?


In [206]:
df = children_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [207]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [208]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [209]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,[],US,,1599150603,,B00DU10PUG,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",false,Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],US,,1934876569,,,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",false,Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
2,Dog Heaven,Dog Heaven,[],US,eng,0590417010,,B017RORXNI,9780590417013,86259,89378,193,4.43,1331,"[{'count': '450', 'name': 'to-read'}, {'count'...",false,Hardcover,40,Blue Sky Press,1,9,1995,,In Newbery Medalist Cynthia Rylant's classic b...,"[{'author_id': '5411', 'role': ''}]","[834493, 452189, 140185, 1897316, 2189812, 424..."
3,"Moths and Mothers, Feathers and Fathers: A Sto...","Moths and Mothers, Feathers and Fathers: A Sto...",[],US,,0915190575,,,9780915190577,3242879,3209312,4,4.29,11,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,,,,,,,,,"[{'author_id': '589328', 'role': ''}, {'author...",[]
4,What Do You Do?,What Do You Do?,[],US,,1416904999,,,9781416904991,1695373,1698376,4,3.57,23,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,Board Book,24,Little Simon,1,6,2005,,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,"[{'author_id': '169159', 'role': ''}]",[]


In [210]:
df['clean_title'] = df['title_without_series']

### Detecting languages

In [211]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [212]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 72496/72496 [02:20<00:00, 516.38it/s]


In [213]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,[],US,,1599150603,,B00DU10PUG,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",false,Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[],The Aeneid for Boys and Girls,eng,The Aeneid for Boys and Girls Relates in vigor...,eng
1,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],US,,1934876569,,,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",false,Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216...",All's Fairy in Love and War (Avalon: Web of Ma...,eng,All's Fairy in Love and War (Avalon: Web of Ma...,eng
2,Dog Heaven,Dog Heaven,[],US,eng,0590417010,,B017RORXNI,9780590417013,86259,89378,193,4.43,1331,"[{'count': '450', 'name': 'to-read'}, {'count'...",false,Hardcover,40,Blue Sky Press,1,9,1995,,In Newbery Medalist Cynthia Rylant's classic b...,"[{'author_id': '5411', 'role': ''}]","[834493, 452189, 140185, 1897316, 2189812, 424...",Dog Heaven,eng,Dog Heaven In Newbery Medalist Cynthia Rylant'...,nan
3,"Moths and Mothers, Feathers and Fathers: A Sto...","Moths and Mothers, Feathers and Fathers: A Sto...",[],US,,0915190575,,,9780915190577,3242879,3209312,4,4.29,11,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,,,,,,,,,"[{'author_id': '589328', 'role': ''}, {'author...",[],"Moths and Mothers, Feathers and Fathers: A Sto...",eng,"Moths and Mothers, Feathers and Fathers: A Sto...",eng
4,What Do You Do?,What Do You Do?,[],US,,1416904999,,,9781416904991,1695373,1698376,4,3.57,23,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,Board Book,24,Little Simon,1,6,2005,,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,"[{'author_id': '169159', 'role': ''}]",[],What Do You Do?,eng,What Do You Do? WHAT DO YOU DO?\nA hen lays eg...,eng


In [214]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [215]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [216]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
2,Dog Heaven,[],eng,0590417010,9780590417013,86259,89378,193,4.43,1331,"[{'count': '450', 'name': 'to-read'}, {'count'...",Hardcover,40,Blue Sky Press,1,9,1995,,In Newbery Medalist Cynthia Rylant's classic b...,"[{'author_id': '5411', 'role': ''}]","[834493, 452189, 140185, 1897316, 2189812, 424..."
3,"Moths and Mothers, Feathers and Fathers: A Sto...",[],eng,0915190575,9780915190577,3242879,3209312,4,4.29,11,"[{'count': '8', 'name': 'to-read'}, {'count': ...",,,,,,,,,"[{'author_id': '589328', 'role': ''}, {'author...",[]
4,What Do You Do?,[],eng,1416904999,9781416904991,1695373,1698376,4,3.57,23,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Board Book,24,Little Simon,1,6,2005,,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,"[{'author_id': '169159', 'role': ''}]",[]


In [217]:
df.shape

(124082, 21)

In [218]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [219]:
df.shape

(104064, 21)

In [220]:
df['format'].value_counts().shape

(148,)

In [221]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['Paperback', 'Hardcover', '', 'Board Book', 'Audiobook', 'Audio', 'Unknown Binding', 'Audio CD', 'Mass Market Paperback', 'Library Binding', 'ebook', 'Board book', 'Boxed Set', 'Kindle Edition', 'Novelty Book', 'School &amp; Library Binding', 'Spiral-bound', 'Audio Cassette', 'Online', 'paperback', 'Audible Audio', 'Leather Bound', 'Hardcover, illustrated', 'MP3 CD', 'Bath Book', 'Turtleback', 'audio cassette', 'unknown', 'a3', 'Broche', 'hardcover', 'Part-In-Omnibus', 'audio download', 'Hardback', 'Trade Paperback', 'Novelty', "Publisher's Binding", 'Hardcover with Lift and Learn Surprises', 'MP3 Audio', 'Paperback + CD', 'Paperback &amp; Hardcover', 'Nook', 'Misc. Supplies', 'MP3 Book', 'Cloth Bound', 'hardbound', 'audio', 'cloth', 'hardback', 'Softcover', 'Picture/Flip Book', 'Paper', 'Soft cover', 'Paperback with CD', 'picture book', 'Board', 'Comic', 'Rag Book', 'DVD', 'paper', 'hard cover', 'Playaway', 'Fabric Bound', 'Book', 'A Big Little Book', 'Cloth Book', 'Padded Hardback',

In [222]:
# 1. Define readable formats to keep
keep_formats = {
    # Paperback
    "Paperback", "paperback", "Mass Market Paperback", "Trade Paperback",
    "Perfect Paperback", "Softcover", "Soft cover", "soft cover", "Soft Cover",
    "Paper", "paper", "Broche", "Uncorrected Proof, paperback",

    # Hardcover / hardback
    "Hardcover", "hardcover", "Hardback", "hardback", "Hard Cover", "hard cover",
    "Hardbound", "hardbound", "Hardcover, illustrated",
    "Hardcover with Dust Jacket", "Padded Hardback", "Cloth Bound", "cloth",
    "Clothbound Hardback", "Hardback, cloth", "Fabric Bound",
    "Library Bound hardback", "Publisher's Binding", "Gebundene Ausgabe",

    # Board / children's readable books
    "Board Book", "Board book", "Board", "Boardbook", "Abridged Board Book",
    "Board book, Hardcover", "Bath Book", "Rag Book", "Cloth Book",
    "Foam Book", "cardbord",

    # Ebook / readable digital text
    "ebook", "Kindle Edition", "Nook", "free ebook", "digital", "Online",

    # Picture books / comics / novelty readable books
    "Comic", "Comic Book", "Hardcover Comic", "Picture Book", "picture book",
    "Picture/Flip Book", "Hardcover Picture Book", "Children's picture book",
    "Little Golden Book", "Big Book", "Novelty Book", "Novelty",
    "Hardcover Novelty Book", "Pop-up", "pop up book", "Pop Up Book",
    "Pop-Up Book", "Pop-up Book", "Hardcover Pop-Up",
    "Hardcover (Pop-Up Boxed Set)", "Box-Fold-Out", "A Big Little Book",
    "Flexibound",

    # Bindings that are still readable books
    "Library Binding", "School &amp; Library Binding",
    "Library Binding, Hardcover", "Turtleback", "Textbook Binding",
    "Reinforced binding", "Leather Bound", "Leatherette softcover",
    "Spiral-bound", "Spiral Binding in Hardcover", "Staple bound",
    "Binder-Bound", "Unbound", "Pamphlet",

    # Box sets / collections
    "Boxed Set", "Paperback Boxed Set", "three books in boxed set",
    "Paperback, Hardcover", "Paperback &amp; Hardcover", "Part-In-Omnibus",
    "Book", "Other Format", "Collector's Edition", "Advanced Reader Copy",
    "Fanfiction"
}

# 2. Define formats to drop
drop_formats = {
    # Audio
    "Audiobook", "Audio", "audio", "Audible Audio", "Audio CD", "audio CD",
    "Audio CD (Unabridged)", "Audio Cassette", "audio cassette",
    "audio download", "MP3 CD", "MP3 Audio", "MP3 Book", "Digital Audio",
    "Digital audio player", "Preloaded Digital Audio Player", "Playaway",
    "Playaway Audiobook", "eAudiobook",

    # Mixed audio/CD bundles
    "Paperback + CD", "Paperback with CD", "Paperback with CD-Rom",
    "Paperback, Audio CD", "Book + Cassette", "Book + Audio Cassette",
    "Book and CD", "Hardcover and CD", "Hardcover with CD",
    "Hardcover + CD", "Hardcover + 2 CDs", "Hardcover + Audiobook",
    "Audio CD + ebook", "audiobook/ebook",

    # Apps / video / software
    "iPad and iPhone app", "DVD", "CD-ROM",

    # Non-book / unclear products
    "Misc. Supplies", "Cards", "Covercraft", "Album", "Issue", "a3",
    "Stories of Anton Checkhov Collection",

    # unknown
    "unknown", "Unknown Binding"
}

# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      85392
review    15561
other      3111
Name: count, dtype: int64

In [223]:
df = df[df["format_group"] == "text"].copy()

In [224]:
df["format_group"].value_counts()

format_group
text    85392
Name: count, dtype: int64

In [225]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[],Paperback,text
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216...",Paperback,text
2,Dog Heaven,[],eng,0590417010,9780590417013,86259,89378,193,4.43,1331,"[{'count': '450', 'name': 'to-read'}, {'count'...",Hardcover,40,Blue Sky Press,1,9,1995,,In Newbery Medalist Cynthia Rylant's classic b...,"[{'author_id': '5411', 'role': ''}]","[834493, 452189, 140185, 1897316, 2189812, 424...",Hardcover,text
4,What Do You Do?,[],eng,1416904999,9781416904991,1695373,1698376,4,3.57,23,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Board Book,24,Little Simon,1,6,2005,,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,"[{'author_id': '169159', 'role': ''}]",[],Board Book,text
5,It's Funny Where Ben's Train Takes Him,[],eng,0531301060,9780531301067,2613165,2592648,3,3.68,21,"[{'count': '10', 'name': 'to-read'}, {'count':...",Hardcover,,Orchard Books (NY),1,3,1999,,Ben draws a train that takes him to all sorts ...,"[{'author_id': '4918', 'role': ''}]",[],Hardcover,text


In [226]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [227]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [228]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,2006,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,2009,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
2,Dog Heaven,[],eng,0590417010,9780590417013,86259,89378,193,4.43,1331,"[{'count': '450', 'name': 'to-read'}, {'count'...",Hardcover,40,Blue Sky Press,1995,In Newbery Medalist Cynthia Rylant's classic b...,"[{'author_id': '5411', 'role': ''}]","[834493, 452189, 140185, 1897316, 2189812, 424..."
4,What Do You Do?,[],eng,1416904999,9781416904991,1695373,1698376,4,3.57,23,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Board Book,24,Little Simon,2005,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,"[{'author_id': '169159', 'role': ''}]",[]
5,It's Funny Where Ben's Train Takes Him,[],eng,0531301060,9780531301067,2613165,2592648,3,3.68,21,"[{'count': '10', 'name': 'to-read'}, {'count':...",Hardcover,,Orchard Books (NY),1999,Ben draws a train that takes him to all sorts ...,"[{'author_id': '4918', 'role': ''}]",[]


In [229]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [230]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [231]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [232]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [233]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,81,Five Little Peppers Abroad,"[{'author_id': '5315', 'role': ''}]",[[238328]],eng,[1426450796],[9781426450792],[7918],[BiblioLife],[2008.0],This is a pre-1923 historical reproduction tha...,"[{'count': '167', 'name': 'to-read'}, {'count'...","[[856114, 832252, 831193, 141305, 179576, 7901...",11,215,3.78,NaN
1,84,Baby Island,"[{'author_id': '5325', 'role': ''}, {'author_i...",[],eng,"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...","[1993.0, 1965.0, 1966.0]",Not many years ago one of the favorite activit...,"[{'count': '692', 'name': 'to-read'}, {'count'...","[[421643, 128521, 3788, 334811, 3797, 826852, ...",182,1315,4.04,172.0
2,115,Where the Red Fern Grows,"[{'author_id': '6810', 'role': ''}]",[],eng,"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","[1961.0, 1996.0, 1998.0, 1974.0, 1986.0, 2003....","For fans ofOld Yeller andShiloh, Where the Red...","[{'count': '2851', 'name': 'classics'}, {'coun...","[[207153, 564858, 130580, 232109, 819138, 7746...",7868,271951,4.04,376.0
3,120,"Drina Goes on Tour (Drina, #10)","[{'author_id': '227840', 'role': 'Pseudonym'},...",[[144393]],eng,[0750002468],[9780750002462],[3155],[Simon and Schuster],[1991.0],Drina passes her school exams and becomes a se...,"[{'count': '69', 'name': 'to-read'}, {'count':...",[],7,111,4.20,188.0
4,121,"Drina Dances Again (Drina, #5)","[{'author_id': '227840', 'role': ''}]",[[144394]],eng,"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],"[1960.0, 1989.0]",A sprained ankle at the start of term forces D...,"[{'count': '79', 'name': 'to-read'}, {'count':...","[[1521492, 706154, 842452, 3780, 3791, 1363190...",6,209,4.07,192.0


In [234]:
df = books_work_df.copy()

In [235]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,81,Five Little Peppers Abroad,"[{'author_id': '5315', 'role': ''}]",[[238328]],eng,[1426450796],[9781426450792],[7918],[BiblioLife],[2008.0],This is a pre-1923 historical reproduction tha...,"[{'count': '167', 'name': 'to-read'}, {'count'...","[[856114, 832252, 831193, 141305, 179576, 7901...",11,215,3.78,NaN
1,84,Baby Island,"[{'author_id': '5325', 'role': ''}, {'author_i...",[],eng,"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...","[1993.0, 1965.0, 1966.0]",Not many years ago one of the favorite activit...,"[{'count': '692', 'name': 'to-read'}, {'count'...","[[421643, 128521, 3788, 334811, 3797, 826852, ...",182,1315,4.04,172.0
2,115,Where the Red Fern Grows,"[{'author_id': '6810', 'role': ''}]",[],eng,"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","[1961.0, 1996.0, 1998.0, 1974.0, 1986.0, 2003....","For fans ofOld Yeller andShiloh, Where the Red...","[{'count': '2851', 'name': 'classics'}, {'coun...","[[207153, 564858, 130580, 232109, 819138, 7746...",7868,271951,4.04,376.0
3,120,"Drina Goes on Tour (Drina, #10)","[{'author_id': '227840', 'role': 'Pseudonym'},...",[[144393]],eng,[0750002468],[9780750002462],[3155],[Simon and Schuster],[1991.0],Drina passes her school exams and becomes a se...,"[{'count': '69', 'name': 'to-read'}, {'count':...",[],7,111,4.20,188.0
4,121,"Drina Dances Again (Drina, #5)","[{'author_id': '227840', 'role': ''}]",[[144394]],eng,"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],"[1960.0, 1989.0]",A sprained ankle at the start of term forces D...,"[{'count': '79', 'name': 'to-read'}, {'count':...","[[1521492, 706154, 842452, 3780, 3791, 1363190...",6,209,4.07,192.0


In [236]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description              2879
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                4751
dtype: int64

In [237]:
df.shape

(64242, 17)

In [238]:
df = df[df["description"].notna()].copy()

In [239]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description                 0
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                4108
dtype: int64

In [240]:
df.shape

(61363, 17)

In [241]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,81,Five Little Peppers Abroad,"[{'author_id': '5315', 'role': ''}]",[[238328]],eng,[1426450796],[9781426450792],[7918],[BiblioLife],[2008.0],This is a pre-1923 historical reproduction tha...,"[{'count': '167', 'name': 'to-read'}, {'count'...","[[856114, 832252, 831193, 141305, 179576, 7901...",11,215,3.78,NaN
1,84,Baby Island,"[{'author_id': '5325', 'role': ''}, {'author_i...",[],eng,"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...","[1993.0, 1965.0, 1966.0]",Not many years ago one of the favorite activit...,"[{'count': '692', 'name': 'to-read'}, {'count'...","[[421643, 128521, 3788, 334811, 3797, 826852, ...",182,1315,4.04,172.0
2,115,Where the Red Fern Grows,"[{'author_id': '6810', 'role': ''}]",[],eng,"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","[1961.0, 1996.0, 1998.0, 1974.0, 1986.0, 2003....","For fans ofOld Yeller andShiloh, Where the Red...","[{'count': '2851', 'name': 'classics'}, {'coun...","[[207153, 564858, 130580, 232109, 819138, 7746...",7868,271951,4.04,376.0
3,120,"Drina Goes on Tour (Drina, #10)","[{'author_id': '227840', 'role': 'Pseudonym'},...",[[144393]],eng,[0750002468],[9780750002462],[3155],[Simon and Schuster],[1991.0],Drina passes her school exams and becomes a se...,"[{'count': '69', 'name': 'to-read'}, {'count':...",[],7,111,4.20,188.0
4,121,"Drina Dances Again (Drina, #5)","[{'author_id': '227840', 'role': ''}]",[[144394]],eng,"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],"[1960.0, 1989.0]",A sprained ankle at the start of term forces D...,"[{'count': '79', 'name': 'to-read'}, {'count':...","[[1521492, 706154, 842452, 3780, 3791, 1363190...",6,209,4.07,192.0


In [242]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [243]:
df.shape

(61363, 15)

In [244]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,81,Five Little Peppers Abroad,"[{'author_id': '5315', 'role': ''}]",[[238328]],[1426450796],[9781426450792],[7918],[BiblioLife],This is a pre-1923 historical reproduction tha...,"[{'count': '167', 'name': 'to-read'}, {'count'...","[[856114, 832252, 831193, 141305, 179576, 7901...",11,215,3.78,NaN
1,84,Baby Island,"[{'author_id': '5325', 'role': ''}, {'author_i...",[],"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...",Not many years ago one of the favorite activit...,"[{'count': '692', 'name': 'to-read'}, {'count'...","[[421643, 128521, 3788, 334811, 3797, 826852, ...",182,1315,4.04,172.0
2,115,Where the Red Fern Grows,"[{'author_id': '6810', 'role': ''}]",[],"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","For fans ofOld Yeller andShiloh, Where the Red...","[{'count': '2851', 'name': 'classics'}, {'coun...","[[207153, 564858, 130580, 232109, 819138, 7746...",7868,271951,4.04,376.0
3,120,"Drina Goes on Tour (Drina, #10)","[{'author_id': '227840', 'role': 'Pseudonym'},...",[[144393]],[0750002468],[9780750002462],[3155],[Simon and Schuster],Drina passes her school exams and becomes a se...,"[{'count': '69', 'name': 'to-read'}, {'count':...",[],7,111,4.20,188.0
4,121,"Drina Dances Again (Drina, #5)","[{'author_id': '227840', 'role': ''}]",[[144394]],"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],A sprained ankle at the start of term forces D...,"[{'count': '79', 'name': 'to-read'}, {'count':...","[[1521492, 706154, 842452, 3780, 3791, 1363190...",6,209,4.07,192.0


In [245]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [246]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [247]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [248]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [249]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [250]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [251]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '167', 'name': 'to-read'}, {'count': '10', 'name': 'classics'}, {'count': '10', 'name': 'fiction'}, {'count': '6', 'name': 'kindle'}, {'count': '5', 'name': 'childrens'}, {'count': '4', 'name': 'children-s'}, {'count': '4', 'name': 'children'}, {'count': '3', 'name': 'kids'}, {'count': '3', 'name': 'historical-fiction'}, {'count': '3', 'name': 'currently-reading'}, {'count': '2', 'name': 'classic'}, {'count': '2', 'name': 'young-adult'}, {'count': '1', 'name': 'owned-kindle-books'}, {'count': '1', 'name': 'good-and-beautiful'}, {'count': '1', 'name': 'ya-children-lit'}, {'count': '1', 'name': 'ya-children-s-lit'}, {'count': '1', 'name': 'victorian-era'}, {'count': '1', 'name': 'the-grand-tour'}, {'count': '1', 'name': 'teen-protagonists'}, {'count': '1', 'name': 'sentimental'}, {'count': '1', 'name': 'road-trip'}, {'count': '1', 'name': 'rich-people-solve-everything'}, {'count': '1', 'name': 'europe'}, {'count': '1', 'name': 'child-protagonists'}, {'count': '1', 'nam

In [252]:
int_df = df[['popular_shelves']]

In [253]:
int_df.head()

,popular_shelves
0,"[{'count': '167', 'name': 'to-read'}, {'count'..."
1,"[{'count': '692', 'name': 'to-read'}, {'count'..."
2,"[{'count': '2851', 'name': 'classics'}, {'coun..."
3,"[{'count': '69', 'name': 'to-read'}, {'count':..."
4,"[{'count': '79', 'name': 'to-read'}, {'count':..."


In [254]:
int_df.shape

(61363, 1)

In [255]:
int_df.to_csv('../data/intermediate/shelves.csv', index=False)

In [ ]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/tag_mapping.csv')

In [257]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,15743699,60570
1,picture-books,picture-book,keep,832907,35597
2,currently-reading,status,drop,622672,33464
3,childrens,children,keep,569261,45725
4,children,children,keep,385575,40705
5,fiction,fiction,keep,373496,30698
6,children-s-books,children,keep,353217,38354
7,favorites,review,drop,331806,18783
8,children-s,children,keep,322927,37874
9,picture-book,picture-book,keep,266242,28316


In [258]:
vocab_df = shelves.copy()

In [259]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [260]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,15743699,60570
1,picture-books,picture-book,keep,832907,35597
2,currently-reading,status,drop,622672,33464
3,childrens,children,keep,569261,45725
4,children,children,keep,385575,40705


In [261]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [262]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [263]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '167', 'name': 'to-read'}, {'count'...","[{'count': '10', 'name': 'classics'}, {'count'..."
1,"[{'count': '692', 'name': 'to-read'}, {'count'...","[{'count': '51', 'name': 'children'}, {'count'..."
2,"[{'count': '2851', 'name': 'classics'}, {'coun...","[{'count': '2851', 'name': 'classics'}, {'coun..."
3,"[{'count': '69', 'name': 'to-read'}, {'count':...","[{'count': '7', 'name': 'ballet'}, {'count': '..."
4,"[{'count': '79', 'name': 'to-read'}, {'count':...","[{'count': '12', 'name': 'children'}, {'count'..."


In [264]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '167', 'name': 'to-read'}, {'count': '10', 'name': 'classics'}, {'count': '10', 'name': 'fiction'}, {'count': '6', 'name': 'kindle'}, {'count': '5', 'name': 'childrens'}, {'count': '4', 'name': 'children-s'}, {'count': '4', 'name': 'children'}, {'count': '3', 'name': 'kids'}, {'count': '3', 'name': 'historical-fiction'}, {'count': '3', 'name': 'currently-reading'}, {'count': '2', 'name': 'classic'}, {'count': '2', 'name': 'young-adult'}, {'count': '1', 'name': 'owned-kindle-books'}, {'count': '1', 'name': 'good-and-beautiful'}, {'count': '1', 'name': 'ya-children-lit'}, {'count': '1', 'name': 'ya-children-s-lit'}, {'count': '1', 'name': 'victorian-era'}, {'count': '1', 'name': 'the-grand-tour'}, {'count': '1', 'name': 'teen-protagonists'}, {'count': '1', 'name': 'sentimental'}, {'count': '1', 'name': 'road-trip'}, {'count': '1', 'name': 'rich-people-solve-everything'}, {'count': '1', 'name': 'europe'}, {'count': '1', 'name': 'child-protagonists'}, {

In [265]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [266]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [267]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '242', 'name': 'children'}, {'count': '32', 'name': 'fiction'}, {'count': '22', 'name': 'adventure'}, {'count': '23', 'name': 'young-adult'}, {'count': '17', 'name': 'classics'}, {'count': '8', 'name': 'read-aloud'}, {'count': '6', 'name': 'early-reader'}, {'count': '7', 'name': 'middle-grade'}, {'count': '4', 'name': 'family'}, {'count': '4', 'name': 'historical-fiction'}, {'count': '3', 'name': 'contemporary'}, {'count': '3', 'name': 'fantasy'}, {'count': '2', 'name': 'mystery'}]


In [268]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '226', 'name': 'fantasy'}
{'count': '149', 'name': 'young-adult'}
{'count': '80', 'name': 'fiction'}
{'count': '349', 'name': 'children'}
{'count': '37', 'name': 'middle-grade'}
{'count': '38', 'name': 'historical-fiction'}
{'count': '28', 'name': 'fairy-tales'}
{'count': '9', 'name': 'magic'}
{'count': '9', 'name': 'adventure'}
{'count': '17', 'name': 'science-fiction'}
{'count': '7', 'name': 'early-reader'}
{'count': '10', 'name': 'classics'}
{'count': '5', 'name': 'family'}
{'count': '4', 'name': 'paranormal'}


In [269]:
df.shape

(61363, 18)

In [270]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,81,Five Little Peppers Abroad,"[{'author_id': '5315', 'role': ''}]",[238328],[1426450796],[9781426450792],[7918],[BiblioLife],This is a pre-1923 historical reproduction tha...,"[{'count': '167', 'name': 'to-read'}, {'count'...","[856114, 832252, 831193, 141305, 179576, 7901,...",11,215,3.78,NaN,[5315],1,"[{'count': '15', 'name': 'classics'}, {'count'..."
1,84,Baby Island,"[{'author_id': '5325', 'role': ''}, {'author_i...",[],"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...",Not many years ago one of the favorite activit...,"[{'count': '692', 'name': 'to-read'}, {'count'...","[421643, 128521, 3788, 334811, 3797, 826852, 4...",182,1315,4.04,172.0,"[5325, 867185]",0,"[{'count': '242', 'name': 'children'}, {'count..."
2,115,Where the Red Fern Grows,"[{'author_id': '6810', 'role': ''}]",[],"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","For fans ofOld Yeller andShiloh, Where the Red...","[{'count': '2851', 'name': 'classics'}, {'coun...","[207153, 564858, 130580, 232109, 819138, 77460...",7868,271951,4.04,376.0,[6810],0,"[{'count': '3652', 'name': 'classics'}, {'coun..."
3,120,Drina Goes on Tour,"[{'author_id': '227840', 'role': 'Pseudonym'},...",[144393],[0750002468],[9780750002462],[3155],[Simon and Schuster],Drina passes her school exams and becomes a se...,"[{'count': '69', 'name': 'to-read'}, {'count':...",[],7,111,4.20,188.0,"[227840, 4114801]",0,"[{'count': '14', 'name': 'ballet'}, {'count': ..."
4,121,Drina Dances Again,"[{'author_id': '227840', 'role': ''}]",[144394],"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],A sprained ankle at the start of term forces D...,"[{'count': '79', 'name': 'to-read'}, {'count':...","[1521492, 706154, 842452, 3780, 3791, 13631900...",6,209,4.07,192.0,[227840],0,"[{'count': '50', 'name': 'children'}, {'count'..."


In [271]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [272]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [273]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,81,Five Little Peppers Abroad,[5315],[238328],[1426450796],[9781426450792],[7918],[BiblioLife],This is a pre-1923 historical reproduction tha...,"[856114, 832252, 831193, 141305, 179576, 7901,...",11,215,3.78,NaN,1,"[{'count': '15', 'name': 'classics'}, {'count'..."
1,84,Baby Island,"[5325, 867185]",[],"[0613132513, 0689717512, 059033221X]","[9780613132510, 9780689717512, 9780590332217]","[8534712, 7932, 7934, 16114578]","[Turtleback Books, Aladdin, Scholastic, The Ma...",Not many years ago one of the favorite activit...,"[421643, 128521, 3788, 334811, 3797, 826852, 4...",182,1315,4.04,172.0,0,"[{'count': '242', 'name': 'children'}, {'count..."
2,115,Where the Red Fern Grows,[6810],[],"[0385020597, 0606109722, 0030547741, 055313897...","[9780385020596, 9780606109727, 9780030547744, ...","[1883225, 3159165, 10370, 1762894, 917282, 824...","[Doubleday Books for Young Readers, Demco Medi...","For fans ofOld Yeller andShiloh, Where the Red...","[207153, 564858, 130580, 232109, 819138, 77460...",7868,271951,4.04,376.0,0,"[{'count': '3652', 'name': 'classics'}, {'coun..."
3,120,Drina Goes on Tour,"[227840, 4114801]",[144393],[0750002468],[9780750002462],[3155],[Simon and Schuster],Drina passes her school exams and becomes a se...,[],7,111,4.20,188.0,0,"[{'count': '14', 'name': 'ballet'}, {'count': ..."
4,121,Drina Dances Again,[227840],[144394],"[0340033274, 0590425579]","[9780340033272, 9780590425575]","[12277577, 3150]",[Apple Paperbacks],A sprained ankle at the start of term forces D...,"[1521492, 706154, 842452, 3780, 3791, 13631900...",6,209,4.07,192.0,0,"[{'count': '50', 'name': 'children'}, {'count'..."


In [274]:
df.shape

(61363, 16)

In [275]:
df.to_csv('../data/intermediate/books-cleaned/children.csv', index=False)